# Dataset analysis

Inspect the included demo CSV without modifying it. All displayed values are computed when the cells run.

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "datasets").exists():
    root = root.parent
data_path = root / "datasets" / "autods_demo_sales.csv"
df = pd.read_csv(data_path)
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
print(f"Source: {data_path}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]} | Duplicate rows: {df.duplicated().sum():,}")
df.head()

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "distinct": df.nunique(dropna=True),
})
display(quality)
display(df.describe(include="all").T)

In [ ]:
valid = df.dropna(subset=["Order Date", "Revenue"]).copy()
valid["Revenue"] = pd.to_numeric(valid["Revenue"], errors="coerce")
valid = valid.dropna(subset=["Revenue"])
print("Observed date range:", valid["Order Date"].min(), "to", valid["Order Date"].max())
display(valid.groupby("Category", dropna=False)["Revenue"].agg(["sum", "count", "mean"]).sort_values("sum", ascending=False))
display(valid.groupby("Product Name")["Revenue"].sum().nlargest(10).rename("observed_revenue"))